In [1]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from imblearn.over_sampling import SMOTENC
from catboost import CatBoostClassifier
import optuna
import joblib
import warnings
warnings.filterwarnings('ignore')

c:\Users\SHANKHADEEP\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
df = pd.read_csv("log.csv")
df.shape

(6362620, 11)

In [7]:
df['isFraud'].value_counts()

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [ ]:
# # 1. Load Data from SQLite
# def load_data(db_path="agora_transactions.db"):
#     print("Loading data from database...")
#     conn = sqlite3.connect(db_path)
#     # Pull everything from our established schema
#     df = pd.read_sql_query("SELECT * FROM transactions", conn)
#     conn.close()
#     return df

# 2. Preprocess for Math Engine
def preprocess_data(df):
    print("Addressing extreme class imbalance via undersampling...")
    
    # Separate the minority and majority classes
    df_fraud = df[df['isFraud'] == 1]
    df_normal = df[df['isFraud'] == 0]
    
    print(f"Original Fraud: {len(df_fraud)} | Original Normal: {len(df_normal)}")
    
    # Drastically downsample the normal transactions (e.g., to 82,000 rows to make a 1:10 ratio)
    # This prevents RAM death while giving the model enough normal patterns to learn from
    df_normal_downsampled = df_normal.sample(n=82000, random_state=42)
    
    # Combine them back into a single dataframe
    df_balanced = pd.concat([df_normal_downsampled, df_fraud])
    
    # Shuffle the dataset so the classes aren't clustered
    df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f"New Dataset Size: {len(df_balanced)} rows")
    
    # Target variable
    y = df_balanced['isFraud']
    
    # Drop string features to prevent CatBoost memory crashes
    X = df_balanced.drop(columns=['nameOrig', 'nameDest', 'isFraud', 'isFlaggedFraud'])
    
    return X, y

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from catboost import CatBoostClassifier
import optuna

# 3. Optimize and Train Model (Plan B: Native CatBoost Balancing)
def train_model():
    # Load from CSV for local testing; replace with load_data() for actual DB
    df = pd.read_csv("log.csv")  
    X, y = preprocess_data(df)
    
    # Identify the categorical feature index for CatBoost
    # Our columns are: step, type, amount, oldbalanceOrg, newbalanceOrig, oldbalanceDest, newbalanceDest
    # 'type' is at index 1
    categorical_features_indices = [1]
    
    # Train/Test Split
    print("Splitting data...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    print(f"Training shape: {X_train.shape}")
    print("Bypassing SMOTENC - Utilizing CatBoost Native Balancing (Plan B)...")

    # Optuna Objective Function for Hyperparameter Tuning
    def objective(trial):
        params = {
            "iterations": trial.suggest_int("iterations", 100, 300),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "depth": trial.suggest_int("depth", 4, 8),
            "cat_features": categorical_features_indices,
            "auto_class_weights": "Balanced", # PLAN B: Native class balancing
            "verbose": False,
            "random_seed": 42
        }
        
        model = CatBoostClassifier(**params)
        # We now train directly on the imbalanced X_train, letting CatBoost handle the math
        model.fit(X_train, y_train) 
        preds = model.predict(X_test)
        
        # We optimize for Macro F1 to ensure the minority class (fraud) is caught
        return f1_score(y_test, preds, average="macro")

    print("Running Optuna optimization...")
    # For a local prototype, 5 trials is enough to prove the concept. 
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=5)
    
    print(f"Best Optuna Trial: {study.best_trial.value}")
    print(f"Best Params: {study.best_trial.params}")
    
    # Train final model with best parameters
    print("Training final CatBoost model...")
    best_params = study.best_trial.params
    best_params["cat_features"] = categorical_features_indices
    best_params["auto_class_weights"] = "Balanced" # PLAN B: Ensure final model uses it
    best_params["verbose"] = 100 # Show training progress
    
    final_model = CatBoostClassifier(**best_params)
    final_model.fit(X_train, y_train)
    
    # Evaluate
    print("\n--- Final Model Evaluation ---")
    predictions = final_model.predict(X_test)
    print(classification_report(y_test, predictions))
    
    # Export Model
    model_path = "agora_fraud_model.cbm"
    final_model.save_model(model_path)
    print(f"Model successfully saved to {model_path}")

if __name__ == "__main__":
    train_model()

Addressing extreme class imbalance via undersampling...
Original Fraud: 8213 | Original Normal: 6354407
New Dataset Size: 90213 rows


[I 2026-04-28 21:31:23,095] A new study created in memory with name: no-name-4c50b430-b699-4893-a057-82928a6f80c2


Splitting data...
Training shape: (72170, 7)
Bypassing SMOTENC - Utilizing CatBoost Native Balancing (Plan B)...
Running Optuna optimization...


[I 2026-04-28 21:31:28,699] Trial 0 finished with value: 0.9557244684183706 and parameters: {'iterations': 147, 'learning_rate': 0.09801775955814641, 'depth': 4}. Best is trial 0 with value: 0.9557244684183706.
[I 2026-04-28 21:31:35,119] Trial 1 finished with value: 0.9324035074398727 and parameters: {'iterations': 171, 'learning_rate': 0.030287312952402518, 'depth': 4}. Best is trial 0 with value: 0.9557244684183706.
[I 2026-04-28 21:31:44,155] Trial 2 finished with value: 0.9644138150843768 and parameters: {'iterations': 194, 'learning_rate': 0.08457157124339519, 'depth': 5}. Best is trial 2 with value: 0.9644138150843768.
[I 2026-04-28 21:32:03,367] Trial 3 finished with value: 0.9780237770593856 and parameters: {'iterations': 252, 'learning_rate': 0.0950128967024424, 'depth': 5}. Best is trial 3 with value: 0.9780237770593856.
[I 2026-04-28 21:32:20,890] Trial 4 finished with value: 0.951411965802818 and parameters: {'iterations': 230, 'learning_rate': 0.027869202907121797, 'depth

Best Optuna Trial: 0.9780237770593856
Best Params: {'iterations': 252, 'learning_rate': 0.0950128967024424, 'depth': 5}
Training final CatBoost model...
0:	learn: 0.5457070	total: 60ms	remaining: 15.1s
100:	learn: 0.0279356	total: 6.8s	remaining: 10.2s
200:	learn: 0.0166570	total: 13.6s	remaining: 3.45s
251:	learn: 0.0136751	total: 17s	remaining: 0us

--- Final Model Evaluation ---
              precision    recall  f1-score   support

           0       1.00      0.99      1.00     16400
           1       0.94      1.00      0.97      1643

    accuracy                           0.99     18043
   macro avg       0.97      1.00      0.98     18043
weighted avg       0.99      0.99      0.99     18043

Model successfully saved to agora_fraud_model.cbm


In [11]:
X, y = preprocess_data(df)
    
# Identify the categorical feature index for CatBoost
# Our columns are: step, type, amount, oldbalanceOrg, newbalanceOrig, oldbalanceDest, newbalanceDest
# 'type' is at index 1
categorical_features_indices = [1]

# Train/Test Split
print("Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

Addressing extreme class imbalance via undersampling...
Original Fraud: 8213 | Original Normal: 6354407
New Dataset Size: 90213 rows
Splitting data...


In [12]:
X_test.to_csv("X_test.csv", index=False)